# Maternal Risk Stratification - Exploratory Data Analysis (EDA)
**Goal:** To understand the distribution of clinical features and identify "Red Flags" that distinguish High-Risk patients from Low-Risk patients.

In [9]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import os

# We use 'r' before the path to fix the 'unicodeescape' error you encountered
# Using a relative path is safer for team collaboration
data_path = r'C:\\Users\\USER\\maternal-risk-stratification-ml\\data\\processed\\maternal_health_clean.csv'

df = pd.read_csv(data_path)
print("Data loaded successfully!")
df.head()

Data loaded successfully!


,age,pulse_rate,haemoglobin,gestational_age,systolic_bp,diastolic_bp,bmi,malaria_rdt,miscarriage_history,alcohol_use,hiv_status,risk
0,18,112.0,11.0,21.0,119.0,65.0,30.991736,0.0,0,0,0,1
1,18,76.0,10.3,25.0,104.0,68.0,24.323229,0.0,0,0,0,1
2,24,76.0,10.3,25.0,104.0,68.0,26.284467,0.0,0,0,0,1
3,18,99.0,8.6,20.0,125.0,65.0,24.524346,0.0,0,0,0,1
4,20,76.0,11.6,21.0,112.0,68.0,32.713499,0.0,0,0,0,0


## 1. Data Integrity & Missingness
Before we trust the numbers, we check if any information is missing. A "clean" heatmap means we have a complete picture of every patient.  

In [10]:
# Create a visual map of empty spots
fig_missing = px.imshow(df.isnull(), 
                        labels=dict(x="Health Indicators", y="Patient Record", color="Is Missing?"),
                        title="Data Health Map",
                        color_continuous_scale='Viridis')
fig_missing.show()

## 2. Risk Distribution
We need to know how many patients are 'High Risk' vs 'Low Risk'. If we have too few of one group, the AI might not learn correctly.

In [11]:
risk_counts = df['risk'].value_counts().reset_index()
risk_counts.columns = ['Risk Level', 'Count']
risk_counts['Risk Level'] = risk_counts['Risk Level'].map({0: 'Low Risk', 1: 'High Risk'})

px.bar(risk_counts, x='Risk Level', y='Count', color='Risk Level', 
       title="Class Balance", color_discrete_map={'Low Risk': 'green', 'High Risk': 'red'})

## 3. Clinical Red Flags (Bivariate Analysis)
 This is the most important part for the analysis. We compare key health numbers against the Risk Level.
**Key Question:** Do High-Risk patients have significantly different Blood Pressure, Haemoglobin, or BMI?

In [12]:
# Comparing Systolic BP, Haemoglobin, and BMI
clinical_cols = ['systolic_bp', 'haemoglobin', 'bmi']

for col in clinical_cols:
    fig = px.box(df, x='risk', y=col, color='risk',
                 title=f"Distribution of {col.upper()} by Risk Group",
                 labels={'risk': '0: Low Risk, 1: High Risk'},
                 points="outliers") # Focus on the extreme cases
    fig.show()

## 4. Final Summary Table
A mathematical "cheat sheet" showing the averages and the spread ($IQR$) for our team report.

In [13]:
cont_features = ['age', 'bmi', 'systolic_bp', 'diastolic_bp', 'pulse_rate', 'haemoglobin', 'gestational_age']
summary = df[cont_features].describe().transpose()

# Calculate Interquartile Range (IQR)
summary['IQR'] = summary['75%'] - summary['25%']

print("--- CLINICAL SUMMARY STATISTICS ---")
summary[['mean', '50%', 'std', 'min', 'max', 'IQR']]

--- CLINICAL SUMMARY STATISTICS ---


,mean,50%,std,min,max,IQR
age,26.217652,25.000000,6.712065,15.000000,50.000000,10.00000
bmi,25.314587,24.341758,5.485401,12.345679,49.382716,6.59783
systolic_bp,112.608415,112.000000,9.567463,70.000000,175.000000,13.00000
diastolic_bp,67.855917,68.000000,7.766813,45.000000,99.000000,11.00000
pulse_rate,78.561523,76.000000,10.171733,48.000000,131.000000,11.00000
haemoglobin,11.578902,11.600000,1.599430,4.400000,19.000000,2.00000
gestational_age,21.571630,21.000000,7.352480,4.000000,42.000000,11.00000
